# Import packages

In [1]:

import glob
import os.path
import numpy as np
import mne
import matplotlib.pyplot as plt
import pandas as pd
import yasa

plt.style.use('default')
plt.rc('font', family='Arial')
plt.rc('axes', unicode_minus=False)

mne.set_log_level('CRITICAL')

In [2]:
import dataset

df_info = pd.read_excel('info.xlsx', index_col=0)

In [3]:
animal = '0629#'
session = '2024-03-09'
file = os.path.join(dataset.path['tmp'], animal, session, 'raw.edf')

raw = mne.io.read_raw_edf(file, preload=True, verbose=False)
raw.filter(l_freq=0.1, h_freq=30)
group = df_info.loc[animal, 'Genotype']

print(animal, session, group)

In [4]:
raw.set_channel_types({'Loco': 'bio'})
raw.notch_filter(np.arange(50, 100, 50))

thr = 4
annot_muscle, scores_muscle = mne.preprocessing.annotate_muscle_zscore(
    raw,
    ch_type='eeg',
    threshold=thr,
    min_length_good=0.5,
    filter_freq=[110, 120],
)


In [5]:
acc = raw.get_data(picks='Loco', units='uV').squeeze()

flag_bad = np.abs(acc) > 100 * np.ones(shape=acc.shape)
bi_bad = np.zeros(shape=acc.shape)
bi_bad[flag_bad] = 1
bi_bad = np.diff(bi_bad)
bad_start_idx = np.argwhere(bi_bad == 1).squeeze()
bad_end_idx = np.argwhere(bi_bad == -1).squeeze()
sf = raw.info['sfreq']
onsets = bad_start_idx / sf - 1
durations = (bad_end_idx - bad_start_idx) / sf + 1
descriptions = ['Bad loco'] * len(bad_start_idx)
annot_loco = mne.Annotations(
    onsets, durations, descriptions, orig_time=raw.info["meas_date"]
)



In [6]:
spike_events = mne.preprocessing.find_eog_events(
    raw, ch_name=raw.ch_names[0], 
    event_id=1, 
    thresh=300 * 1e-6,
    l_freq=0.5, 
    h_freq=30,
    reject_by_annotation=True
)
onsets = spike_events[:, 0] / raw.info["sfreq"] - 1
durations = [3] * len(spike_events)
descriptions = ["spike"] * len(spike_events)
annot_spike = mne.Annotations(
    onsets, durations, descriptions, orig_time=raw.info["meas_date"]
)

In [7]:
raw.set_annotations(raw.annotations + annot_spike + annot_loco + annot_muscle)
raw.export(raw.filenames[0].replace('.edf', '_spike_dect.edf'), overwrite=True)

In [8]:
epochs = mne.Epochs(raw, spike_events, tmin=-1, tmax=2, event_id={'spike': 1}, preload=True)
epochs.plot_image(picks=raw.ch_names[0], vmin=-800, vmax=800)
plt.show()

In [9]:
peak_amp = epochs.average().get_data(units='uV', picks=raw.ch_names[0]).max()
spike_rate = len(epochs) / raw.times[-1]
print(f'Spike rate: {spike_rate:.2f} s^-1; Peak amplitude: {peak_amp:.2f} μV')

# Multi-sessions

In [10]:
files = glob.glob(os.path.join(dataset.path['tmp'], '**', 'raw.edf'), recursive=True)


In [29]:
for file in files:
    animal, session = np.array(file.split('\\'))[[-3, -2]]
    group = df_info.loc[animal, 'Genotype']
    file = os.path.join(dataset.path['tmp'], animal, session, 'raw.edf')
    raw = mne.io.read_raw_edf(file, preload=True, verbose=False)
    
    raw.set_channel_types({'Loco': 'bio'})
    raw.notch_filter(np.arange(50, 100, 50))
    
    # Muscle activity
    thr = 4
    annot_muscle, scores_muscle = mne.preprocessing.annotate_muscle_zscore(
        raw,
        ch_type='eeg',
        threshold=thr,
        min_length_good=0.5,
        filter_freq=[110, 120],
    )
    
    # Locomotion
    acc = raw.get_data(picks='Loco', units='uV').squeeze()
    
    flag_bad = np.abs(acc) > 100 * np.ones(shape=acc.shape)
    bi_bad = np.zeros(shape=acc.shape)
    bi_bad[flag_bad] = 1
    bi_bad[0] = 0
    bi_bad[-1] = 0
    bi_bad = np.diff(bi_bad)
        
    bad_start_idx = np.argwhere(bi_bad == 1).reshape(-1)
    bad_end_idx = np.argwhere(bi_bad == -1).reshape(-1)   
    if bad_start_idx.shape[0] == 0:
        continue 
    
    sf = raw.info['sfreq']
    onsets = bad_start_idx / sf - 1
    durations = (bad_end_idx - bad_start_idx) / sf + 2
    descriptions = ['BAD_loco'] * len(bad_start_idx)
    annot_loco = mne.Annotations(
        onsets, durations, descriptions, orig_time=raw.info["meas_date"]
    )
    
    # spike detection
    spike_events = mne.preprocessing.find_eog_events(
        raw, ch_name=raw.ch_names[0],
        event_id=1,
        thresh=300 * 1e-6,
        l_freq=0.5,
        h_freq=30,
        reject_by_annotation=True
    )
    if spike_events.shape[0] == 0:
        continue
        
    onsets = spike_events[:, 0] / raw.info["sfreq"] - 1
    durations = [3] * len(spike_events)
    descriptions = ["spike"] * len(spike_events)
    annot_spike = mne.Annotations(
        onsets, durations, descriptions, orig_time=raw.info["meas_date"]
    )
    
    raw.set_annotations(raw.annotations + annot_spike + annot_loco + annot_muscle)
    raw.export(raw.filenames[0].replace('.edf', '_spike_dect.edf'), overwrite=True)
    epochs = mne.Epochs(raw, spike_events, tmin=-1, tmax=2, event_id={'spike': 1}, preload=True)
    epochs.plot_image(picks=raw.ch_names[0], vmin=-800, vmax=800)
    plt.show()
    
    peak_amp = epochs.average().get_data(units='uV', picks=raw.ch_names[0]).max()
    spike_rate = len(epochs) / raw.times[-1]
    print(animal, session, group)
    print(f'Spike rate: {spike_rate:.3f} s^-1; Peak amplitude: {peak_amp:.2f} μV') 

发现效果不是特别好，主要是peak识别的算法不佳，打算使用`find_peaks`重新做一遍。